# 12a · From retrieval hits to traceable context

## The engineering question
Which retrieved text can safely become evidence, and how much fits after counting citation headers?

**Learning outcomes:** validate each hit against an authorized active source snapshot; budget with the generator's actual tokenizer; map request-local citation labels to source revisions/hashes/spans; distinguish citation resolution from support.

Retrieval returns candidates. Evidence assembly verifies source identity, rejects changed text, removes duplicate chunks, and packs complete spans under a fixed token budget. Short labels such as E1 are local to one request; the sidecar maps them to immutable provenance. Model-generated annotations never become canonical source text. The complete chat prompt needs additional tokens beyond this evidence-only budget, enforced by the generator in lesson 12b.

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from vector_masterclass.lesson import LessonRun
from vector_masterclass.config import root

plt.rcParams.update(
    {"figure.figsize": (8, 4), "axes.spines.top": False, "axes.spines.right": False}
)
lesson = LessonRun("12a", "T", docker=False)
print("Kernel:", sys.executable)
print("Workspace:", root())

In [ ]:
from transformers import AutoTokenizer
from vector_masterclass.config import model_path
from vector_masterclass.data import documents, queries
from vector_masterclass.chunking import chunk_document
from vector_masterclass.models import embed
from vector_masterclass.retrieval import exact_search
from vector_masterclass.context import assemble, citation_audit
from vector_masterclass.metrics import evidence_coverage
from vector_masterclass.evidence import json_write

tokenizer = AutoTokenizer.from_pretrained(str(model_path("generator")), local_files_only=True)
docs = documents(tenant="public")
chunks = [c for doc in docs for c in chunk_document(doc)]
lookup = {c.id: c for c in chunks}
vectors = embed([c.text for c in chunks])
question = queries("development")[0]
ranked = exact_search(embed([question.text], role="query")[0], vectors, list(lookup), k=10)
retrieved = [lookup[i] for i, s in ranked]
rows = []
for budget in [32, 64, 128, 256]:
    packed = assemble(retrieved + retrieved[:1], docs, tokenizer, budget, tenant="public")
    rows.append(
        {
            "budget": budget,
            "actual_tokens": packed["tokens"],
            "chunks": len(packed["chunks"]),
            "evidence_coverage": evidence_coverage(packed["chunks"], question.evidence)["coverage"],
        }
    )
json_write(lesson.path / "citations.json", packed["citations"])
print("Question:", question.text)
print(packed["text"])
# Deliberately forged evidence must fail before it can enter a prompt.
forged_rejected = False
try:
    assemble(
        [retrieved[0].model_copy(update={"text": "Unsupported replacement text"})],
        docs,
        tokenizer,
        256,
        tenant="public",
    )
except ValueError:
    forged_rejected = True
audit = citation_audit(
    "A claim that could be wrong [E1]. An invented citation [E999].", packed["citations"]
)
print("Referential audit (not semantic support):", audit)
results = lesson.table(pd.DataFrame(rows))
display(results)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(results.budget, results.actual_tokens, "o-")
axes[0].plot(results.budget, results.budget, "--", color="gray", label="Budget")
axes[0].set(
    xlabel="Evidence token budget", ylabel="Packed tokens", title="Whole spans and citation headers"
)
axes[0].legend()
axes[1].plot(results.budget, results.evidence_coverage, "o-")
axes[1].set(
    xlabel="Evidence token budget",
    ylabel="Gold evidence coverage",
    ylim=(-0.05, 1.05),
    title="More context need not add evidence",
)
fig.tight_layout()
lesson.figure(fig)
plt.show()

## Interpretation and exercise
Packing is discontinuous: an entire chunk either fits or is skipped. A smaller top-ranked chunk can improve utilization, but cutting a qualifier can destroy meaning. Evidence coverage measures supplied source spans, while a resolving citation only proves the identifier exists. Neither guarantees that a generated claim is true or supported.

**Exercise:** halve the evidence budget and inspect the exact source spans lost. Then add a stale revision to the hit list and explain where current-version authorization belongs. Design a policy for partially fitting chunks that preserves offsets and sentence boundaries rather than silently truncating source text.

The snapshot must itself come from an authoritative policy/source layer. If that snapshot is stale, validation against it cannot detect a newly revoked permission. The serving and security lessons address this boundary and document race windows.

In [ ]:
manifest = lesson.finish(
    {
        "budget_respected": bool((results.actual_tokens <= results.budget).all()),
        "no_duplicate_chunks": len(packed["chunks"]) == len({c.id for c in packed["chunks"]}),
        "forged_text_rejected": forged_rejected,
        "unknown_citation_detected": audit["unknown"] == ["E999"],
        "all_citations_have_provenance": all(
            c["document_key"] and c["source_hash"] and c["start"] < c["end"]
            for c in packed["citations"].values()
        ),
    }
)

## References and next steps
Consult the [Qdrant documentation](https://qdrant.tech/documentation/),
the [MiniLM model card](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2),
and [uv's Jupyter guide](https://docs.astral.sh/uv/guides/integration/jupyter/).
The exact pinned configuration is in `configs/models.json`; worked exercises are in `docs/SOLUTIONS.md`.
This notebook's measurements apply to the recorded local run, not a production deployment.